# Experiment 9 — NLP: 20 Newsgroups Classification

This notebook performs a full text classification workflow on the 20 Newsgroups dataset using TF-IDF features and a Logistic Regression classifier. It includes data exploration, preprocessing, training, evaluation, and visualization.

In [ ]:
# Install missing packages when running in Colab
# Uncomment the following line in Colab if packages are missing:
# !pip install -r requirements.txt


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Download NLTK resources used in this notebook
nltk.download('stopwords')
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))


In [ ]:
# Load data
categories = None  # set to a list of category names to restrict the dataset
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers','footers','quotes'))
tests = newsgroups.data
labels = newsgroups.target
label_names = newsgroups.target_names

print(f'Number of documents: {len(texts)}')
print(f'Number of categories: {len(label_names)}')


In [ ]:
# Quick exploration
from collections import Counter
lengths = [len(t.split()) for t in texts]
print('Median tokens per doc:', np.median(lengths))
print('Average tokens per doc:', np.mean(lengths))
print('Top 5 categories by document count:')
print(Counter(label_names[i] for i in labels).most_common()[:5])


In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
print('Train size:', len(X_train), 'Test size:', len(X_test))


In [ ]:
# Simple text cleaner (optional)
def clean_text(doc):
    # basic cleaning: lowercase and remove extra whitespace. Keep it minimal to avoid losing signal.
    return ' '.join(doc.lower().split())

# Apply cleaning lazily inside pipeline; not applied here to raw texts.


In [ ]:
# Build pipeline: TF-IDF -> Logistic Regression
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_df=0.85, min_df=2, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=2000, solver='saga', n_jobs=-1, C=1.0, random_state=42))
])

print('Fitting pipeline (this may take a few minutes)...')
pipe.fit(X_train, y_train)
print('Done.')


In [ ]:
# Evaluate
y_pred = pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Test accuracy:', acc)
print('
Classification report:')
print(classification_report(y_test, y_pred, target_names=label_names))


In [ ]:
# Confusion matrix (all categories)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, cmap='Blues', xticklabels=label_names, yticklabels=label_names, fmt='d')
plt.title('Confusion matrix (all categories)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()


In [ ]:
# Save model
joblib.dump(pipe, 'exp9_tfidf_logreg_20newsgroups.joblib')
print('Saved trained pipeline to exp9_tfidf_logreg_20newsgroups.joblib')


## Next steps
- Hyperparameter tuning (GridSearchCV)
- Try other models (SVM, Random Forest, or fine-tuned transformers)
- Use more advanced preprocessing (lemmatization, domain stopwords)
- Explore class-wise errors and per-category performance
